# Create Train Test Split

### Stratified by label and item difficulty

In [1]:
import pandas as pd 

In [ ]:
human_annotations = pd.read_csv(
    'C:/Users/yhyah/Documents/data/Raters Datasets/New Coding Scheme_Annotated Datasets/LLM Classification/Human Annotations Compared.csv',
    keep_default_na=False,
    na_values=[''],
)
human_annotations

,id,text,obj_index,object,Y_label,E_label,M_label
0,1293669089987239936,@MyDailyCapital @LauraKi8833 @NickAdamsinUSA W...,1,Harris,NA,NA,NA
1,1293669089987239936,@MyDailyCapital @LauraKi8833 @NickAdamsinUSA W...,2,tRump,D,D,D
2,1396597726586966016,@Build_Blue_Wall @andrewsaundry You do underst...,1,Bernie,NA,NA,NA
3,1396597726586966016,@Build_Blue_Wall @andrewsaundry You do underst...,2,Trump,NA,NA,NA
4,1250189212638314496,RT @Acosta: After Fauci clarifies comments Tru...,1,president,NA,NA,NA
...,...,...,...,...,...,...,...
1394,1212624809764540416,RT @fred_guttenberg: Follow this thread. It i...,2,Trump,NA,NA,NA
1395,1274811368714178560,@devinfoley8625 @Qwarantinebored @Bri871Ed @sp...,1,@realDonaldTrump,NA,NA,NA
1396,1274811368714178560,@devinfoley8625 @Qwarantinebored @Bri871Ed @sp...,2,@JoeBiden,NA,NA,NA
1397,1248765009527574528,@Joe_Friedman_ @AgiaTheBun @WagsKoop @SteFonzi...,1,@JoeBiden,NA,NA,NA


### Generate Gold Labels

In [3]:
# generate gold label through majority voting
def majority_label(row):
    counts = row.value_counts()
    if counts.empty or counts.iloc[0] < 2:
        return pd.NA
    return counts.index[0]

human_annotations['gold_label'] = human_annotations[['Y_label', 'E_label', 'M_label']].apply(majority_label, axis=1)
human_annotations

,id,text,obj_index,object,Y_label,E_label,M_label,gold_label
0,1293669089987239936,@MyDailyCapital @LauraKi8833 @NickAdamsinUSA W...,1,Harris,NA,NA,NA,NA
1,1293669089987239936,@MyDailyCapital @LauraKi8833 @NickAdamsinUSA W...,2,tRump,D,D,D,D
2,1396597726586966016,@Build_Blue_Wall @andrewsaundry You do underst...,1,Bernie,NA,NA,NA,NA
3,1396597726586966016,@Build_Blue_Wall @andrewsaundry You do underst...,2,Trump,NA,NA,NA,NA
4,1250189212638314496,RT @Acosta: After Fauci clarifies comments Tru...,1,president,NA,NA,NA,NA
...,...,...,...,...,...,...,...,...
1394,1212624809764540416,RT @fred_guttenberg: Follow this thread. It i...,2,Trump,NA,NA,NA,NA
1395,1274811368714178560,@devinfoley8625 @Qwarantinebored @Bri871Ed @sp...,1,@realDonaldTrump,NA,NA,NA,NA
1396,1274811368714178560,@devinfoley8625 @Qwarantinebored @Bri871Ed @sp...,2,@JoeBiden,NA,NA,NA,NA
1397,1248765009527574528,@Joe_Friedman_ @AgiaTheBun @WagsKoop @SteFonzi...,1,@JoeBiden,NA,NA,NA,NA


In [9]:
# gold label distribution
human_annotations['gold_label'].value_counts(dropna=False)

gold_label
NA      1088
D        270
T         40
<NA>       1
Name: count, dtype: int64

In [10]:
# which row has gold label == <NA> ? (genuine missing, not the label 'NA')\n",
human_annotations[human_annotations['gold_label'].isna()]

,id,text,obj_index,object,Y_label,E_label,M_label,gold_label
997,1437233571954053120,RT @KamalaHarris: Gov. @GavinNewsom is a natio...,2,@GavinNewsom,T,D,NA,<NA>


### Add Item Difficulty

Item difficulty (named 'entropy') was calculated by using the MACE algorithm by Hovy https://github.com/dirkhovy/MACE

In [4]:
# read file
entropies_path = (
    "compared_Entropies.entropies"
)
with open(entropies_path) as f:
    item_difficulties = pd.Series(
        [float(line.strip()) for line in f if line.strip()],
        name="item difficulties",
    )
item_difficulties


0       0.000577
1       0.000735
2       0.000577
3       0.000577
4       0.000577
          ...   
1394    0.000577
1395    0.000577
1396    0.000577
1397    0.000577
1398    0.000735
Name: item difficulties, Length: 1399, dtype: float64

In [5]:
# give it a header and attach it to the dataset
human_annotations['item_difficulty'] = item_difficulties
human_annotations

,id,text,obj_index,object,Y_label,E_label,M_label,gold_label,item_difficulty
0,1293669089987239936,@MyDailyCapital @LauraKi8833 @NickAdamsinUSA W...,1,Harris,NA,NA,NA,NA,0.000577
1,1293669089987239936,@MyDailyCapital @LauraKi8833 @NickAdamsinUSA W...,2,tRump,D,D,D,D,0.000735
2,1396597726586966016,@Build_Blue_Wall @andrewsaundry You do underst...,1,Bernie,NA,NA,NA,NA,0.000577
3,1396597726586966016,@Build_Blue_Wall @andrewsaundry You do underst...,2,Trump,NA,NA,NA,NA,0.000577
4,1250189212638314496,RT @Acosta: After Fauci clarifies comments Tru...,1,president,NA,NA,NA,NA,0.000577
...,...,...,...,...,...,...,...,...,...
1394,1212624809764540416,RT @fred_guttenberg: Follow this thread. It i...,2,Trump,NA,NA,NA,NA,0.000577
1395,1274811368714178560,@devinfoley8625 @Qwarantinebored @Bri871Ed @sp...,1,@realDonaldTrump,NA,NA,NA,NA,0.000577
1396,1274811368714178560,@devinfoley8625 @Qwarantinebored @Bri871Ed @sp...,2,@JoeBiden,NA,NA,NA,NA,0.000577
1397,1248765009527574528,@Joe_Friedman_ @AgiaTheBun @WagsKoop @SteFonzi...,1,@JoeBiden,NA,NA,NA,NA,0.000577


In [13]:
# which values can item_difficulty take?
human_annotations['item_difficulty'].value_counts()

# higher item difficulty values = more uncertainty/disagreement among annotators
# lower item difficulty values = high agreement, easy cases

item_difficulty
0.000577    1007
0.000735     201
0.090620      33
0.066912      30
0.000391      25
0.222686      20
0.165088      19
0.216576      19
0.286676      16
0.120200       7
0.087340       6
0.248930       6
0.037718       4
0.246270       2
0.085312       2
0.074332       1
0.978744       1
Name: count, dtype: int64

In [6]:
# remove the one gold label = NA case (see above)
human_annotations = human_annotations.dropna(subset=['gold_label'])
human_annotations

,id,text,obj_index,object,Y_label,E_label,M_label,gold_label,item_difficulty
0,1293669089987239936,@MyDailyCapital @LauraKi8833 @NickAdamsinUSA W...,1,Harris,NA,NA,NA,NA,0.000577
1,1293669089987239936,@MyDailyCapital @LauraKi8833 @NickAdamsinUSA W...,2,tRump,D,D,D,D,0.000735
2,1396597726586966016,@Build_Blue_Wall @andrewsaundry You do underst...,1,Bernie,NA,NA,NA,NA,0.000577
3,1396597726586966016,@Build_Blue_Wall @andrewsaundry You do underst...,2,Trump,NA,NA,NA,NA,0.000577
4,1250189212638314496,RT @Acosta: After Fauci clarifies comments Tru...,1,president,NA,NA,NA,NA,0.000577
...,...,...,...,...,...,...,...,...,...
1394,1212624809764540416,RT @fred_guttenberg: Follow this thread. It i...,2,Trump,NA,NA,NA,NA,0.000577
1395,1274811368714178560,@devinfoley8625 @Qwarantinebored @Bri871Ed @sp...,1,@realDonaldTrump,NA,NA,NA,NA,0.000577
1396,1274811368714178560,@devinfoley8625 @Qwarantinebored @Bri871Ed @sp...,2,@JoeBiden,NA,NA,NA,NA,0.000577
1397,1248765009527574528,@Joe_Friedman_ @AgiaTheBun @WagsKoop @SteFonzi...,1,@JoeBiden,NA,NA,NA,NA,0.000577


In [15]:
# save
human_annotations.to_csv('Human Annotations with Gold labels and item difficulty.csv', index = True)

## Create stratified train test split

Stratified by label and item difficulty

In [7]:
# write a custom function for stratification by multiple variables
# function is deterministic; does not require a set seed
import numpy as np
from collections import Counter
from typing import List


def stratified_split(df: pd.DataFrame, num_folds: int, variables: List[str]):
    df = df.copy()
    df = df.sort_values(variables)
    df['fold'] = np.arange(len(df)) % num_folds

    return [df[df['fold'] == fold].drop(columns=['fold']) for fold in np.arange(num_folds)]

folds = stratified_split(human_annotations, 2, ['gold_label', 'item_difficulty'])

for i, fold in enumerate(folds):
    print(f'Fold {i} (n={len(fold)})')
    print(f'gold_label: {Counter(fold["gold_label"])}')
    print(f'Mean item_difficulty: {np.mean(fold["item_difficulty"]):.2f}')


Fold 0 (n=699)
gold_label: Counter({'NA': 544, 'D': 135, 'T': 20})
Mean item_difficulty: 0.02
Fold 1 (n=699)
gold_label: Counter({'NA': 544, 'D': 135, 'T': 20})
Mean item_difficulty: 0.02


Train and Test dataset match in distribution of gold labels and item difficulty

In [9]:
train = folds[0]
test = folds[1]

train

,id,text,obj_index,object,Y_label,E_label,M_label,gold_label,item_difficulty
1,1293669089987239936,@MyDailyCapital @LauraKi8833 @NickAdamsinUSA W...,2,tRump,D,D,D,D,0.000735
16,1362503852117929984,@JSCCounterPunch @PattyArquette This story abo...,2,Ted Cruz,D,D,D,D,0.000735
18,1334769141794832384,@4everpeggy @Rothbard1776 @SidneyPowell1 I hav...,2,Trump,D,D,D,D,0.000735
88,1465483301599203328,BREAKING NEWS: The January 6 Committee announc...,3,Trump,D,D,D,D,0.000735
103,1353849346622558208,I’d like to know if Steven Mnuchin will be inv...,2,Steven Mnuchin,D,D,D,D,0.000735
...,...,...,...,...,...,...,...,...,...
780,1430568005554032640,"After a rigorous process, the FDA reaffirmed i...",1,FDA,T,T,NA,T,0.120200
328,1241445759272333312,The best @JoeBiden add to date!\r\n\r\nA brief...,1,@JoeBiden,NA,T,T,T,0.246270
128,1351954863962390528,Congratulations President Biden and Vice Presi...,1,President Biden,T,NA,T,T,0.248930
130,1351954863962390528,Congratulations President Biden and Vice Presi...,3,@POTUS,T,NA,T,T,0.248930


In [11]:
# save train and test
train.to_csv("train.csv", index=False)
test.to_csv("test.csv", index=False)